# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

In [3]:
# langsmith for tracing and evals
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("LangSmith API Key:")


os.environ["LANGCHAIN_PROJECT"] = "AIE9 11_Advanced Retrieval"

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [5]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch**: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog**: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts**: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds. Repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [18]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Adequate and quality sleep—typically 7-9 hours per night—supports physical health by allowing the body to repair tissues, regulate hormones, and strengthen the immune system. It also enhances mental well-being, cognitive functions such as memory and learning, and mood stability. Poor sleep or sleep disorders like insomnia can negatively affect these processes, leading to increased health risks, weakened immunity, and mental health issues. Maintaining good sleep hygiene and creating an optimal sleep environment are essential strategies for promoting overall health.'

In [19]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using peppermint or lavender essential oils\n- Maintaining a regular sleep schedule\n- Practicing deep breathing, progressive muscle relaxation, or grounding techniques\n- Taking short walks in nature\n- Listening to calming music\n\nThese approaches can help alleviate stress and headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [11]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [12]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [13]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (cat position) and letting it sag down (cow position). Do 10-15 repetitions.\n\n2. **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold each for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle stretches and strengthening movements that may help alleviate lower back discomfort and prevent future episodes. Always consult with a healthcare professional before starting new exercises, especially if you have existing back issues.'

In [14]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Sleep plays a crucial role in overall health. Adequate and quality sleep, typically 7-9 hours per night for adults, supports the body's natural healing and regeneration processes, especially during deep sleep stages. It helps regulate mood, memory, and cognitive function, and contributes to a strong immune system. Maintaining a consistent sleep schedule and creating an optimal sleep environment—such as a cool, dark, and quiet room with a comfortable mattress—can improve sleep quality. Conversely, issues like insomnia can negatively impact health, emphasizing the importance of good sleep habits for overall wellness."

In [14]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root may help reduce stress and promote relaxation. Additionally, ensuring proper hydration, managing sleep hygiene, and avoiding headache triggers such as certain foods can also be beneficial.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

BM25 would be useful fo use cases where we're more concerned with matching the exact word (the lexical pattern) rather than just similar words (semantic similarity). It might be good for searching product names they are unique, e.g. looking for product descriptions. Example: "Does Ninja AF400UK Foodi MAX Air Fryer have two drawers?". Here I don't care about air fryers in general, I want a description of the exact product. Semantic search with embeddings could return documents about another very similar air fryer because it doesn't do the keyword search, whilst BM25 will surface references with the exact name of the product since it uses a deterministic algorithm.

So product sales would be a good use case, but also in healthcare (medication names are unique, e.g. various painkiller meds can use the same or similar ingredients but have a totally different name). I could see the legal sector would be another good domain since legal documents are all about precise wording.

PS This reminds me of regex as regex is a formal language (form - lexis vs meaning - semantics). Regex is used for normalization and for pattern matching, e.g. to accept specific sequences such as phone numbers, IDs. This took me down a little rabbit hole and I found out that regex can be helpful in the RAG pipelines for not only normalization of the user query before its sent to the retriever but also for PII redaction like here https://www.elastic.co/search-labs/blog/rag-security-masking-pii 


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [14]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [15]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [16]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, here are some exercises that can help with lower back pain:\n\n1. Cat-Cow Stretch: Start on your hands and knees. Alternate between arching your back up (cat position) and letting it sag down (cow position). Perform 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Aim for 10 repetitions per side.\n\n3. Pelvic Tilts: Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nAlways consult with a healthcare professional before starting new exercises, especially if you have ongoing back pain.'

In [18]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours for adults, involves cycles of REM and non-REM stages, which are essential for restorative processes. Creating an optimal sleep environment—such as maintaining a cool, dark, quiet, and comfortable setting—can enhance sleep quality. Poor sleep or conditions like insomnia can negatively affect health, highlighting the importance of good sleep habits for overall wellness.'

In [19]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include drinking water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gently massaging the temples and neck, using peppermint or lavender essential oils, maintaining a regular sleep schedule, practicing deep breathing and progressive muscle relaxation, engaging in grounding techniques, taking short walks in nature, and listening to calming music.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [16]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [17]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back upward (cat) and letting it sag downward (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for 5 seconds and switch sides. Aim for 10 repetitions per side.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Hold briefly and then lower back down. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times

In [34]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical recovery, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate and quality sleep—typically 7-9 hours per night—also contributes to a stronger immune system, better emotional regulation, and overall longevity. Poor sleep or sleep disturbances, such as insomnia, can negatively impact physical health, increase stress, and reduce mental clarity. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are essential for promoting overall health and well-being.'

In [35]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (inhale for 4 counts, hold for 4, exhale for 4)\n- Progressive muscle relaxation (tensing and releasing muscle groups)\n- Grounding techniques (naming things you see, hear, feel, smell, and taste)\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck\n- Using essential oils like peppermint or lavender\n\nThese methods can help provide immediate relief and promote overall relaxation.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

We can say the same thing in a myriad of ways, that's the beauty and power of language (in linguistics, that's the domain of pragmatics - how we express meaning - and sociolinguistics - how language use varies speaker by speaker). By generating multiple reformulations of a user query, we are tapping into many more linguistic forms that express very similar meaning, and hence, expand the list of references that can be used for response generation.

That's why in the classic NLU-driven chatbots a human (like me) would curate a set of training phrases to express a user intention that could be then used to train an ML classifier to recognise the user's input. And that's why it's also important to understand your user - to know what they need but also to know (more or less) how they ask for what they want. Still, the possibilities are technichcally endless.

PS Language models of today are great at the formal aspects of language (syntax, semantics) but not so great at the functional aspects of it. That's because humans are physical beings acting in the real-world using language in an embodied way, drawing on a suite of  cognitive skills, world knowledge representations, schemas and multi-modal context. <end_of_language_nerd_rant> :)

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [18]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [19]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [20]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [21]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [22]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [26]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, some gentle stretching and strengthening exercises are recommended. These include:\n\n- Cat-Cow Stretch: On hands and knees, alternate arching your back up (cat) and letting it sag down (cow), doing 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged, hold for 5 seconds, then switch sides. Perform 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat, hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, tighten your abs and tilt your pelvis slightly to flatten your back against the floor, hold for 10 seconds, repeat 8-12 times.\n\nAlways remember to perform these exercises gently and consult with a health

In [42]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health in several ways. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7-9 hours of sleep per night. Good sleep quality supports healthy immune function, maintains mood stability, and enhances concentration and learning. Poor sleep or sleep disturbances can lead to increased risk of health issues such as fatigue, headaches, impaired cognitive function, and emotional stress. Therefore, maintaining healthy sleep habits and hygiene is vital for overall wellness.'

In [43]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, engaging in progressive muscle relaxation, doing grounding exercises, taking short walks especially in nature, listening to calming music, and practicing mindfulness or meditation. For headaches specifically, natural remedies include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gently massaging the temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [23]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [24]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [29]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.  \n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.  \n- Pelvic Tilts: Lie on your back with knees bent, tighten your abs and tilt your pelvis up slightly to flatten your back against the floor. Hold for 10 seconds and repeat 8-12 times.  \n- Partial Crunches: Lie on your back with knees bent, cross arms over your chest, and gently lift your shoulders off the floor while tightening your stomach muscles. Hold briefly, then lower. Do 8-12 repetitions.  \n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.  \n\nThese exercises are gentle stretching and strengthening mov

In [47]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical repair, mental well-being, and cognitive functions such as memory and learning. During sleep, tissues are repaired, hormones that regulate growth and appetite are released, and the brain consolidates memories. Adequate sleep (7-9 hours per night) supports immune function, reduces stress, and boosts mental health. Poor sleep or sleep disturbances like insomnia can negatively affect physical health, mental well-being, and increase vulnerability to illness. Maintaining good sleep hygiene—such as keeping a consistent schedule, creating a comfortable sleep environment, and practicing relaxation techniques—can help promote better sleep quality and, consequently, improve overall health.'

In [48]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques (such as naming things you see, hear, feel, smell, and taste), taking short walks especially in nature, and listening to calming music. \n\nFor headaches, natural remedies include staying well-hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gentle massage of the temples and neck, using essential oils such as peppermint or lavender, maintaining a regular sleep schedule, and using caffeine in small amounts if appropriate.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [25]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [26]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [27]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [28]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [29]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [35]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting the pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises are recommended to alleviate discomfort and prevent future episodes of lower back pain.'

In [56]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health because it allows the body to repair tissues, regulate hormones, and support cognitive functions like memory and learning. Adequate sleep (7-9 hours per night) helps maintain physical health, mental well-being, and proper immune function. Poor sleep or disruptions in sleep cycles can lead to issues such as fatigue, headaches, weakened immune response, and impaired mental health. Good sleep hygiene practices and creating an optimal sleep environment are important for ensuring restful sleep and promoting overall wellness.'

In [57]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4, and hold for 4.\n- Progressive muscle relaxation: Tense and relax muscle groups, such as from toes to head.\n- Grounding techniques: Name 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste.\n- Taking short walks, preferably in nature.\n- Listening to calming music.\n- Applying peppermint or lavender essential oils.\n- Staying well-hydrated by drinking water.\n- Resting in a dark, quiet room.\n- Gentle massage of temples and neck.\n\nThese approaches can help reduce tension, promote relaxation, and alleviate headache symptoms naturally.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

If sentences are highly repetetive such as in FAQs, semantic chunking might cram many utterances into one chunk and fail to see subtle differences between them. Hence, a percentile breakpointer might chunk in the wrong places, that is, it might make big chunks or draw potentially arbitrary boundaries because it wouldn't "see" big semantic jumps between seperate FAQs.

Firstly, I would consider using a fixed size chunking method since in FAQs sentences are structured similarly and might easily yield to splitting on common patterns such as question marks. Secondly, I would explore topic modelling with clustering for chunking. I say this based on my experience with classic NLU conversational AI systems.

Also found this paper https://arxiv.org/html/2410.13070v1 that mentions semantic chunking is quite costly computationally and not necessarily the best shot, especially in simpler solutions such as basic FAQs :)

The thresholding methods are not explained in this notebook and the link to the LangChain website goes to the Langchain Overview page. I did my own research but didn't find a whole lot on the breakpointers so it makes it harder to extrapolate to this scenario but I tried! 


---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

Setup LangSmith tracing for evals (top of notebook)

Create synthetic dataset with Ragas using abstracted approach 

In [31]:
# reuse loaded data for testset generation
health_wellness_docs = raw_docs

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.testset import TestsetGenerator

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# generate testset
dataset = generator.generate_with_langchain_docs(health_wellness_docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/9 [00:00<?, ?it/s]

In [32]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Wut is the Bird Dog exersize and how do I do it?,[The Personal Wellness Guide A Comprehensive R...,"Bird Dog: From hands and knees, extend opposit...",single_hop_specifc_query_synthesizer
1,Me want know what Personal Wellness Guide say ...,[The Personal Wellness Guide A Comprehensive R...,"The Personal Wellness Guide say for exercise, ...",single_hop_specifc_query_synthesizer
2,What is the recommended way to perform the Kne...,[The Personal Wellness Guide A Comprehensive R...,To perform the Knee-to-Chest Stretch for lower...,single_hop_specifc_query_synthesizer
3,"what chapter 8 say bout sleep, how i make slee...",[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Chapter 8 talk about improving sleep quality. ...,single_hop_specifc_query_synthesizer
4,magnesium help sleep?,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Magnesium supplements is a natural remedy for ...,single_hop_specifc_query_synthesizer
5,"According to Chapter 7, what are the key stage...",[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Chapter 7 explains that sleep is essential for...,single_hop_specifc_query_synthesizer
6,What are some sources of Vitamin D that can he...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"According to the context, sources of Vitamin D...",single_hop_specifc_query_synthesizer
7,What are some dietary sources of zinc that can...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"According to the context, dietary sources of z...",single_hop_specifc_query_synthesizer
8,"Why zinc good for immune, where I get zinc from?",[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Zinc help immune function. You get zinc from o...,single_hop_specifc_query_synthesizer


Define evaluation metrics

In [33]:
from ragas import EvaluationDataset
from ragas.metrics import LLMContextRecall, ContextEntityRecall, NoiseSensitivity, ContextPrecision 
from ragas import evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper

# define metrics
retriever_metrics = [
    LLMContextRecall(),
    ContextPrecision(),
    ContextEntityRecall(),
    NoiseSensitivity(),
]

# define run config
run_config = RunConfig(timeout=360)
# define evaluator
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Evaluate naive retriever

In [34]:
# get context and response from naive retriever
for test_row in dataset:
  response = naive_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"].content
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [35]:
dataset.samples[0].eval_sample.response

'The Bird Dog exercise is a movement that helps strengthen your lower back and core muscles. To do the Bird Dog exercise:\n\n1. Start on your hands and knees on a comfortable, stable surface.\n2. Keep your back flat and core engaged.\n3. Extend your opposite arm and leg simultaneously—for example, raise your right arm straight out in front of you and straighten your left leg behind you.\n4. Keep your hips and shoulders square to the floor, and avoid sagging or arching your back.\n5. Hold this position for about 5 seconds.\n6. Slowly return to the starting position and switch sides, extending your left arm and right leg.\n7. Repeat this movement for about 10 repetitions on each side.\n\nThis exercise helps improve stability, balance, and strengthening of the lower back and core muscles. Remember to perform the movement slowly and with control to avoid strain.'

In [36]:
# convert dataset table into EvaluationDataset
eval_dataset_naive = EvaluationDataset.from_pandas(dataset.to_pandas())
# run evaluation
result_naive = evaluate(
    eval_dataset_naive,
    retriever_metrics,
    evaluator_llm,
    run_config=run_config,
)

result_naive

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception raised in Job[7]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[23]: TimeoutError()


{'context_recall': 0.9444, 'context_precision': 0.6188, 'context_entity_recall': 0.2587, 'noise_sensitivity_relevant': 0.0139}

Evaluate BM25 retriever

In [39]:
# copy dataset
import copy
bm25_dataset = copy.deepcopy(dataset)

# get context and response from bm25 retriever
for test_row in bm25_dataset:
    result = bm25_retrieval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = result["response"].content
    test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in result["context"]]

eval_dataset_bm25 = EvaluationDataset.from_pandas(bm25_dataset.to_pandas())

# evaluate
result_bm25 = evaluate(
    eval_dataset_bm25,
    retriever_metrics,
    evaluator_llm,
    run_config=run_config,
)

result_bm25

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

{'context_recall': 0.4556, 'context_precision': 0.1944, 'context_entity_recall': 0.2384, 'noise_sensitivity_relevant': 0.0028}

Evaluate reranker

In [40]:
rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
    result = contextual_compression_retrieval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = result["response"].content
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in result["context"]]

eval_dataset_rerank = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())
result_rerank = evaluate(
    eval_dataset_rerank,
    retriever_metrics,
    evaluator_llm,
    run_config=run_config,
)

result_rerank

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

{'context_recall': 0.8778, 'context_precision': 0.8889, 'context_entity_recall': 0.4115, 'noise_sensitivity_relevant': 0.0682}

Evaluate Multi-Query Retriever

In [41]:
multi_query_dataset = copy.deepcopy(dataset)

for test_row in multi_query_dataset:
    result = multi_query_retrieval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = result["response"].content
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in result["context"]]

eval_dataset_multi_query = EvaluationDataset.from_pandas(multi_query_dataset.to_pandas())
result_multi_query = evaluate(
    eval_dataset_multi_query,
    retriever_metrics,
    evaluator_llm,
    run_config=run_config,
)

result_multi_query  

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception raised in Job[3]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[23]: TimeoutError()


{'context_recall': 0.9333, 'context_precision': 0.6362, 'context_entity_recall': 0.3881, 'noise_sensitivity_relevant': 0.1476}

Evaluate Parent-Document Retriever

In [42]:
parent_document_dataset = copy.deepcopy(dataset)

for test_row in parent_document_dataset:
    result = parent_document_retrieval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = result["response"].content
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in result["context"]]

eval_dataset_parent_document = EvaluationDataset.from_pandas(parent_document_dataset.to_pandas())
result_parent_document = evaluate(
    eval_dataset_parent_document,
    retriever_metrics,
    evaluator_llm,
    run_config=run_config,
)

result_parent_document

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

{'context_recall': 0.9000, 'context_precision': 0.7778, 'context_entity_recall': 0.4342, 'noise_sensitivity_relevant': 0.0948}

Evaluate Ensemble Retriever

In [43]:
ensemble_dataset = copy.deepcopy(dataset)

for test_row in ensemble_dataset:
    result = ensemble_retrieval_chain.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = result["response"].content
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in result["context"]]

eval_dataset_ensemble = EvaluationDataset.from_pandas(ensemble_dataset.to_pandas())
result_ensemble = evaluate(
    eval_dataset_ensemble,
    retriever_metrics,
    evaluator_llm,
    run_config=run_config,
)

result_ensemble

Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception raised in Job[3]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[23]: TimeoutError()


{'context_recall': 1.0000, 'context_precision': 0.4659, 'context_entity_recall': 0.4288, 'noise_sensitivity_relevant': 0.0133}

Compare performance

In [45]:
type(result_naive)

ragas.dataset_schema.EvaluationResult

In [ ]:
# aggregate results 
import pandas as pd

retrievers_performance_comparison = {
    "naive": result_naive,
    "bm25": result_bm25,
    "rerank": result_rerank,
    "multi_query": result_multi_query,
    "parent_document": result_parent_document,
    "ensemble": result_ensemble,
}

#  mean of each metric 
summary_df = pd.DataFrame(
    {name: pd.DataFrame(res.scores).mean() for name, res in retrievers_performance_comparison.items()}
).T
summary_df

,context_recall,context_precision,context_entity_recall,noise_sensitivity_relevant
naive,0.944444,0.618827,0.258718,0.013889
bm25,0.455556,0.194444,0.238426,0.002849
rerank,0.877778,0.888889,0.411527,0.068160
multi_query,0.933333,0.636243,0.388113,0.147619
parent_document,0.900000,0.777778,0.434238,0.094758
ensemble,1.000000,0.465867,0.428802,0.013333


**Results**

- Reranker performs best on average across all the metrics
  - strong context recall 0.88 and highest context precision of about 0.89 (which means that the most relevant docs are surfaced to the top of references); 
  -  entity recall is on par with other retrievers and it has relatively low noise sensitivity
  -  cost higher than the naive retriever but not as expensive as e.g. ensemble retriever
  
- Parent Document follows second
  - with a high context recall of 0.9  good context precision of 0.78
  -  context entity recall is similar to the reranker
  
- Ensemble retriever 
  - scores a perfect context recall of 1 (!) but has loer precision (0.46) so it seems to be retrieving irrelevant chunks
  - its expensive and slow compared to the more precise reranker
  
- Naive retriever 
  - has a very good context recall of 0.94 although context precision is lower than in other advanced retrievers which is not surprising as no ranking is taking place here.
  - it emerges as a good baseline with low costs

- Multi-query
  - is not a massive improvemt as compared to naive baseline and entails additional costs
  
- BM25
  - is cheap and fast but has the lowest recall and very low precision on this data
  - it has to be noted that in the ensamble it seems to be driving the numbers up though so its probably good in hybrid solutions

In terms of cost and latency, ensemble and multi-query are most expensive which is not surprising since they involve more LLM calls.

The reranker is the best overall retriever for this use case given this quick test. It shows good performance and seems reasonable in terms of costs compared to other advanced retrievers. Naive retriever seems like a good first shot and a baseline for evaluating more advanced methods.